## PDF Extraction

In [1]:
import pdfplumber
import pandas as pd
import os

# Path to the PDF file
pdf_path = "TypenprüfungHIT_MVXAnlagenHP.pdf"
output_dir = "TypenprüfungHIT_MVXAnlagenHP_pages_csv"

# Create the output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Function to extract tables from each page and save as CSV if they have at least 5 rows
def extract_tables_to_csv(pdf_path, output_dir):
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages):
            tables = page.extract_tables()
            if tables:
                for j, table in enumerate(tables):
                    df = pd.DataFrame(table)
                    if len(df) >= 5:  # Check if the table has at least 5 rows
                        output_file = os.path.join(output_dir, f"page_{i+1}_table_{j+1}.csv")
                        df.to_csv(output_file, index=False)

# Extract tables and save as CSV files
extract_tables_to_csv(pdf_path, output_dir)

# List of created CSV files
output_files = os.listdir(output_dir)
output_files

['page_100_table_2.csv',
 'page_10_table_2.csv',
 'page_11_table_2.csv',
 'page_12_table_2.csv',
 'page_13_table_2.csv',
 'page_14_table_2.csv',
 'page_15_table_2.csv',
 'page_16_table_2.csv',
 'page_17_table_2.csv',
 'page_18_table_2.csv',
 'page_19_table_2.csv',
 'page_1_table_2.csv',
 'page_20_table_2.csv',
 'page_21_table_2.csv',
 'page_22_table_2.csv',
 'page_23_table_2.csv',
 'page_24_table_2.csv',
 'page_25_table_2.csv',
 'page_26_table_2.csv',
 'page_27_table_2.csv',
 'page_28_table_2.csv',
 'page_29_table_2.csv',
 'page_2_table_2.csv',
 'page_30_table_2.csv',
 'page_31_table_2.csv',
 'page_32_table_2.csv',
 'page_33_table_2.csv',
 'page_34_table_2.csv',
 'page_35_table_2.csv',
 'page_36_table_2.csv',
 'page_37_table_2.csv',
 'page_38_table_2.csv',
 'page_39_table_2.csv',
 'page_3_table_2.csv',
 'page_40_table_2.csv',
 'page_41_table_2.csv',
 'page_42_table_2.csv',
 'page_43_table_2.csv',
 'page_44_table_2.csv',
 'page_45_table_2.csv',
 'page_46_table_2.csv',
 'page_47_table_2.

## CSV File extraction

In [2]:
import os
import pandas as pd
import numpy as np
import re


import os

# Define the folder path containing the CSV files
folder_path = "TypenprüfungHIT_MVXAnlagenHP_pages_csv"

# Define the processed folder path in the same directory as the script
processed_folder_path = "Processed"

# Create the 'Processed' folder if it doesn't exist
if not os.path.exists(processed_folder_path):
    os.makedirs(processed_folder_path)

print(f"Processed folder path: {processed_folder_path}")


# Function to process each CSV file
def process_csv(file_path):
    df = pd.read_csv(file_path)

    # Replace NaN with None to handle integer checking properly
    df = df.where(pd.notnull(df), None)

    # Extract the base product type
    base_product_type = df.iloc[0, 0][:25]

    # Function to check if a value is an integer
    def is_integer(value):
        try:
            return float(value).is_integer()
        except (ValueError, TypeError):
            return False

    # Find the starting column and row based on the "[mm]" header
    start_col = None
    start_row = None
    for col in df.columns:
        for row in df.index:
            if df.at[row, col] == "[mm]":
                start_col = df.columns.get_loc(col)  # Get the index of the start column
                start_row = row + 1
                break
        if start_col is not None:
            break

    if start_col is not None and start_row is not None:
        # Iterate over the rows starting from the identified starting row
        for index, row in df.iloc[start_row:].iterrows():
            first_col_value = df.at[index, df.columns[start_col]]
            if is_integer(first_col_value):
                first_col_value = int(float(first_col_value))

                for col in df.columns[2:]:  # Start from the second column to the end
                    if row[col] is not None:
                        suffix = ""
                        col_idx = df.columns.get_loc(col)
                        if start_col + 1 <= col_idx <= start_col + 4:  # Adjust based on the start column
                            suffix = " C20/25"
                            value_label = {start_col + 1: "MRD1", start_col + 2: "VRD1", start_col + 3: "MRD2", start_col + 4: "VRD2"}[col_idx]
                        elif start_col + 5 <= col_idx <= start_col + 8:
                            suffix = " C25/30"
                            value_label = {start_col + 5: "MRD1", start_col + 6: "VRD1", start_col + 7: "MRD2", start_col + 8: "VRD2"}[col_idx]
                        elif start_col + 9 <= col_idx <= start_col + 12:
                            suffix = " C30/37"
                            value_label = {start_col + 9: "MRD1", start_col + 10: "VRD1", start_col + 11: "MRD2", start_col + 12: "VRD2"}[col_idx]
                        else:
                            value_label = ""

                        original_value = row[col]
                        processed_value = f"{original_value} h{first_col_value}{suffix} {value_label} CC50"
                        # Extract the numeric value after 'h'
                        num = int(processed_value.split('h')[1].split()[0])
                        # Add the additional hh part
                        processed_value += f" hh{num + 50}"

                        df.at[index, col] = processed_value

    # Replace all occurrences of 'hh' with 'h' in the DataFrame
    df.replace(to_replace='h h', value=' hh', regex=True, inplace=True)

    # Remove the gap between 'h' and the number for all relevant cells
    df = df.applymap(lambda x: x.replace(' h', 'h') if isinstance(x, str) else x)

    # Move MRD/VRD values to the beginning of the string
    def move_mrd_vrd_to_beginning(cell):
        if isinstance(cell, str) and any(label in cell for label in ['MRD', 'VRD']):
            parts = cell.split()
            # Identify the part containing MRD/VRD and move it to the beginning
            for i, part in enumerate(parts):
                if 'MRD' in part or 'VRD' in part:
                    mrd_vrd_part = parts.pop(i)
                    parts.insert(0, mrd_vrd_part)
                    break
            return ' '.join(parts)
        return cell

    df = df.applymap(move_mrd_vrd_to_beginning)

    # Filter rows to keep only those containing "MRD" or "hh"
    rows_to_keep = df.apply(lambda row: row.astype(str).str.contains('MRD|hh').any(), axis=1)
    filtered_df = df[rows_to_keep]

    # Filter columns to keep only those containing "MRD" or "hh"
    cols_to_keep = filtered_df.apply(lambda col: col.astype(str).str.contains('MRD|hh').any())
    filtered_df = filtered_df.loc[:, cols_to_keep]

    # Identify rows to drop (containing 'MVX')
    rows_to_drop = filtered_df.apply(lambda row: row.astype(str).str.contains('MVX').any(), axis=1)

    # Identify columns to drop (containing 'MVX')
    cols_to_drop = filtered_df.apply(lambda col: col.astype(str).str.contains('MVX').any(), axis=0)

    # Drop identified rows and columns
    filtered_df = filtered_df.loc[~rows_to_drop, ~cols_to_drop]

    # Reindex the filtered_df
    filtered_df.reset_index(drop=True, inplace=True)

    # Check if the number of columns is odd
    if filtered_df.shape[1] % 2 != 0:
        # Drop the first column
        filtered_df = filtered_df.drop(columns=filtered_df.columns[0])

    # Reindex columns
    filtered_df.columns = range(filtered_df.shape[1])

    # Create a new DataFrame to hold the combined values
    combined_df = pd.DataFrame()

    # Function to remove duplicates within a cell, considering patterns
    def remove_duplicates(cell):
        parts = cell.split()
        seen = set()
        unique_parts = []
        for part in parts:
            # Identify the base pattern (e.g., "h125") and use it to check for duplicates
            base_pattern = re.sub(r'[\d,.]', '', part)
            if base_pattern not in seen:
                seen.add(base_pattern)
                unique_parts.append(part)
        return ' '.join(unique_parts)

    # Iterate over the columns in pairs
    for i in range(0, len(filtered_df.columns), 2):
        # Combine adjacent columns
        combined_column = filtered_df.iloc[:, i] + ' ' + filtered_df.iloc[:, i + 1]
        # Remove duplicates within each cell
        combined_column = combined_column.apply(remove_duplicates)
        # Add the combined column to the new DataFrame
        combined_df[i // 2] = combined_column

    # Function to parse each cell and extract the required values
    def parse_cell(cell):
        mrd_match = re.search(r'(MRD\d) ([\d,.-]+)h(\d+)', cell)
        vrd_match = re.search(r'(VRD\d) ([\d,.-]+)h(\d+)', cell)
        c_match = re.search(r'C(\d+/\d+)', cell)
        cc_match = re.search(r'CC(\d+)', cell)
        hh_match = re.search(r'hh(\d+)', cell)
        
        mrd_type = mrd_match.group(1)[-1] if mrd_match else None
        mrd_value = mrd_match.group(2) if mrd_match else None
        vrd_type = vrd_match.group(1)[-1] if vrd_match else None
        vrd_value = vrd_match.group(2) if vrd_match else None
        c_value = c_match.group(1) if c_match else None
        cc_value = cc_match.group(1) if cc_match else None
        hh_value = hh_match.group(1) if hh_match else None
        h_value = mrd_match.group(3) if mrd_match else None

        return {
            'MRD': mrd_value,
            'VRD': vrd_value,
            'C': c_value,
            'CC': cc_value,
            'hh': hh_value,
            'h': h_value,
            'MRD_Type': mrd_type,
            'VRD_Type': vrd_type
        }

    # Flatten the combined_df and parse each cell
    parsed_rows = []
    for row in combined_df.itertuples(index=False):
        for cell in row:
            parsed_rows.append(parse_cell(cell))

    # Create a new DataFrame from the parsed rows
    parsed_df = pd.DataFrame(parsed_rows)

    # Define the additional CC values and the corresponding adjustment for hh
    cc_values = [30, 35, 40, 45]

    # Function to create new rows with different CC values
    def create_new_rows_with_different_cc(row):
        new_rows = []
        for cc in cc_values:
            new_row = row.copy()
            new_row['CC'] = cc
            new_row['hh'] = int(new_row['h']) + cc  # Adjust hh value
            new_rows.append(new_row)
        return new_rows

    # Create a list to hold all new rows
    all_rows = []

    # Iterate over each row in the original DataFrame
    for index, row in parsed_df.iterrows():
        all_rows.append(row)  # Keep the original row
        new_rows = create_new_rows_with_different_cc(row)
        all_rows.extend(new_rows)  # Add the new rows with different CC values

    # Convert the list of all rows back into a DataFrame
    expanded_df = pd.DataFrame(all_rows)

    expanded_df['CC'] = expanded_df['CC'].astype(int)
    expanded_df['hh'] = expanded_df['hh'].astype(int)
    expanded_df['h'] = expanded_df['h'].astype(int)

    # Add conditions for CC based on h values
    def filter_cc_based_on_h(df):
        # Drop rows where h is 310 or 320 and CC is not 30
        condition1 = (df['h'].isin([310, 320])) & (df['CC'] != 30)
        # Drop rows where h is 305 or 315 and CC is not 35
        condition2 = (df['h'].isin([305, 315])) & (df['CC'] != 35)
        
        return df[~(condition1 | condition2)]

    # Apply the filter conditions
    expanded_df = filter_cc_based_on_h(expanded_df)

    # Create the 'product_type' column with the base product type
    expanded_df['product_type'] = base_product_type

    # Make all the columns titles lowercase
    expanded_df.columns = map(str.lower, expanded_df.columns)

    # Add the new column 'new_product_type' with hh divided by 10
    expanded_df['new_product_type'] = expanded_df.apply(
        lambda row: row['product_type'].replace('hh', str(row['hh'] / 10)).replace('cc', str(row['cc'])),
        axis=1
    )

    # Replace all the ',' with '.' in the whole dataframe
    expanded_df = expanded_df.applymap(lambda x: x.replace(',', '.') if isinstance(x, str) else x)

    # Save the processed DataFrame to a new CSV file with "_processed" appended to the filename
    base_filename = os.path.basename(file_path)
    new_filename = os.path.splitext(base_filename)[0] + "_processed.csv"
    new_file_path = os.path.join(processed_folder_path, new_filename)
    expanded_df.to_csv(new_file_path, index=False)

# Process all CSV files in the folder
for file_name in os.listdir(folder_path):
    if file_name.endswith(".csv"):
        file_path = os.path.join(folder_path, file_name)
        process_csv(file_path)


Processed folder path: Processed


C:\Users\gabri\AppData\Local\Temp\ipykernel_21580\4272072771.py:87: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.replace(' h', 'h') if isinstance(x, str) else x)
C:\Users\gabri\AppData\Local\Temp\ipykernel_21580\4272072771.py:102: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(move_mrd_vrd_to_beginning)
C:\Users\gabri\AppData\Local\Temp\ipykernel_21580\4272072771.py:248: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  expanded_df = expanded_df.applymap(lambda x: x.replace(',', '.') if isinstance(x, str) else x)
C:\Users\gabri\AppData\Local\Temp\ipykernel_21580\4272072771.py:87: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.replace(' h', 'h') if isinstance(x, str) else x)
C:\Users\gabri\AppData\Local\Temp\ipykernel_21580\4272072771.py:102: FutureWarning: DataFrame.a

## Processing the CSV files

In [3]:
import os
import pandas as pd


# Define the processed folder path directly as separate
processed_folder_path = "Processed"

# Create the 'Processed' folder if it doesn't exist
if not os.path.exists(processed_folder_path):
    os.makedirs(processed_folder_path)

print(f"Processed folder path: {processed_folder_path}")



# Initialize an empty list to hold DataFrames
df_list = []

# Read each processed CSV file and append to the list
for file_name in os.listdir(processed_folder_path):
    if file_name.endswith("_processed.csv"):
        file_path = os.path.join(processed_folder_path, file_name)
        df = pd.read_csv(file_path)
        df_list.append(df)

# Concatenate all DataFrames into one master DataFrame
master_df = pd.concat(df_list, ignore_index=True)

# Drop rows where 'hh' ends with 5
master_df = master_df[~master_df['hh'].astype(str).str.endswith('5')]

# Remove .0 from all string representations in the DataFrame
master_df = master_df.applymap(lambda x: str(x).replace('.0', '') if isinstance(x, str) else x)

# Save the master DataFrame to a new CSV file
master_file_path = os.path.join(processed_folder_path, "masterfile_HIT_HP.csv")
master_df.to_csv(master_file_path, index=False)

# Display the path to the master file
print(f"Master file created at: {master_file_path}")

Processed folder path: Processed


C:\Users\gabri\AppData\Local\Temp\ipykernel_21580\302573866.py:33: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  master_df = master_df.applymap(lambda x: str(x).replace('.0', '') if isinstance(x, str) else x)


Master file created at: Processed\masterfile_HIT_HP.csv


## Finalizing Files

In [4]:
# Replace 'HIT-HP ' with 'HIT_HP-' in the 'new_product_type' column
master_df['new_product_type'] = master_df['new_product_type'].apply(lambda x: x.replace('HIT-HP ', 'HIT_HP-'))

# Define new columns to be extracted
new_columns = ['Thickness', 'Type', 'CSB', 'Height', 'Width', 'CC_Type']

# Split 'new_product_type' into the new columns
master_df[new_columns] = master_df['new_product_type'].str.split('-', expand=True)

# Convert 'Height' from string to integer and multiply by 10
master_df['Height'] = master_df['Height'].astype(int) * 10

# Drop the unnecessary columns
master_df.drop(columns=['product_type', 'Height', 'CC_Type'], inplace=True)

# Reorder columns with 'new_product_type' at the beginning
master_df = master_df[['new_product_type'] + [col for col in master_df.columns if col != 'new_product_type']]

# Rename columns for better readability
master_df.rename(columns={'new_product_type': 'product_name'}, inplace=True)
master_df.rename(columns={'mrd': 'mRd_minus', 'vrd': 'vRd_plus'}, inplace=True)

# Display the first few rows of the dataframe
master_df = master_df[~master_df['cc'].isin([40, 45])]
master_df.head()



,product_name,mRd_minus,vRd_plus,c,cc,hh,h,mrd_type,vrd_type,Thickness,Type,CSB,Width
2,HIT_HP-MVX-1812-16-100-35,-32.5,192.0,20/25,35,160,125,1,1,HIT_HP,MVX,1812,100
7,HIT_HP-MVX-1812-16-100-35,-65.6,70.0,20/25,35,160,125,2,2,HIT_HP,MVX,1812,100
12,HIT_HP-MVX-1812-16-100-35,-38.5,192.0,25/30,35,160,125,1,1,HIT_HP,MVX,1812,100
17,HIT_HP-MVX-1812-16-100-35,-71.6,73.9,25/30,35,160,125,2,2,HIT_HP,MVX,1812,100
22,HIT_HP-MVX-1812-16-100-35,-42.9,192.0,30/37,35,160,125,1,1,HIT_HP,MVX,1812,100


## Saving CSV Files

In [5]:
import os

# Define the file name for the final CSV file
final_file_name = "final_file_extended_columns_HIT_HP.csv"

# Save the final dataframe to a CSV file in the main directory
final_file_path = os.path.join(final_file_name)

master_df.to_csv(final_file_path, index=False)

print(f"Final file created at: {final_file_path}")


Final file created at: final_file_extended_columns_HIT_HP.csv
